---

# 🔧 Mecánico de bar para motores 2 y 4T
## Inteligencia Artificial aplicada al conocimiento técnico

---

Hola

Os voy a presentar un asistente de inteligencia artificial que convierte cientos de páginas de documentación técnica en respuestas instantáneas y precisas.

Voy a explicar cómo se ha construido paso a paso, qué decisiones técnicas se han tomado y por qué, y terminaremos con una demo en vivo del sistema funcionando.

**La presentación tiene 4 partes:**

- **Selección del dominio y datos** — qué documentos y por qué
- **Construcción del pipeline** — cómo se procesa y almacena el conocimiento
- **El agente RAG** — arquitectura, memoria y decisiones de diseño
- **Demo en vivo** — el sistema funcionando

---

## 📂 Paso 1 — Selección del Dominio y Documentos

---

El dominio elegido es **motores de combustión interna**, concretamente motores de 2 y 4 tiempos.

Se seleccionaron 4 documentos PDF con criterios claros:

| Documento | Páginas | Contenido |
|---|---|---|
| Funcionamiento y Preparación Motor 2T | 49 | Mecánica, ciclos, preparación |
| Motores (resumen técnico) | 7 | Fundamentos generales |
| Curso Motor 2 Tiempos | 8 | Teoría del ciclo |
| Motores de Combustión Interna | 94 | Referencia técnica completa |

**Total: 158 páginas — diversidad de contenido, distintos niveles de detalle, mismo dominio.**

El objetivo era que el agente tuviera suficiente base documental para responder preguntas con distintos niveles de profundidad.

---

## ✂️ Paso 2 — Procesamiento de Documentos (Chunking)

---

Los PDFs no se pueden pasar directamente a un modelo de IA — hay que fragmentarlos en trozos manejables.

**Proceso:**

1. Carga de PDFs con `PyPDFLoader` (extrae el texto página a página)
2. División en fragmentos con `RecursiveCharacterTextSplitter`

**Parámetros de chunking:**

```python
chunk_size = 500      # Máximo 500 caracteres por fragmento
chunk_overlap = 50    # 50 caracteres compartidos entre fragmentos contiguos
```

**Resultado:** 158 páginas → **632 fragmentos** indexables

**Decisión de diseño:** El overlap de 50 caracteres evita que un concepto importante quede cortado a la mitad entre dos fragmentos. Un chunk de 500 caracteres es suficientemente grande para dar contexto, y suficientemente pequeño para recuperar información precisa.

---

## 🔤 Paso 3 — Embeddings

---

Los embeddings convierten texto en vectores numéricos para que el sistema pueda comparar significados, no solo palabras.

**Primera opción: Google Gemini Embeddings**

Los requisitos del proyecto especificaban Gemini Embeddings, pero la API devolvía un error `404 NOT_FOUND` en `v1beta`. La funcionalidad no estaba disponible para el nivel de acceso de la cuenta utilizada.

**Solución adoptada: HuggingFace — `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`**

```python
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
```

**Ventajas de esta elección:**
- ✅ Se ejecuta localmente — sin llamadas a API externas
- ✅ Sin costos ni límites de cuota
- ✅ Multilingüe — funciona correctamente en español
- ✅ Ligero — ~50MB, descarga única

El modelo fue validado por el profesor como alternativa válida a Gemini Embeddings.

---

## 🗄️ Paso 4 — Base de Datos Vectorial (ChromaDB)

---

Una vez generados los embeddings, los 632 fragmentos se indexan en **ChromaDB**, una base de datos vectorial que permite búsqueda semántica.

```python
vectorstore = Chroma.from_documents(
    documents=fragmentos,
    embedding=embeddings,
    persist_directory="./chroma_db_motores"
)
```

**Cómo funciona la búsqueda:**

Cuando el usuario hace una pregunta, se convierte también a vector y se buscan los fragmentos más cercanos en el espacio vectorial:

```python
docs = vectorstore.similarity_search(pregunta, k=3)
# Devuelve los 3 fragmentos más semánticamente similares
```

**Decisión de diseño:** `k=3` equilibra contexto suficiente con precisión. Con más fragmentos se añade ruido; con menos, puede faltar información para responder correctamente.

La base de datos se persiste en disco — **la indexación se hace una sola vez** y se reutiliza en todas las ejecuciones posteriores.

---

## 🤖 Paso 5 — Construcción del Agente LangGraph

---

El agente se construye con **LangGraph**, que permite definir flujos explícitos de procesamiento con nodos y aristas.

### Estado del agente

Cada paso del flujo comparte un estado común:

```python
class EstadoAgente(TypedDict):
    pregunta: str               # Pregunta del usuario
    contexto_documentos: list   # Fragmentos recuperados de ChromaDB
    mensajes: list[BaseMessage] # Historial de conversación
    respuesta: str              # Respuesta generada
```

### Nodo 1 — Retrieval

Recibe la pregunta, busca en ChromaDB y devuelve los 3 fragmentos más relevantes.

### Nodo 2 — Generación

Construye el prompt completo (system prompt + contexto + pregunta) y llama a Gemini para generar la respuesta.

### Flujo del grafo

```
START → [Retrieval] → [Generación] → END
```

Dos nodos. Un flujo lineal. Simple y robusto.

---

## 📝 Paso 6 — System Prompt y Temperatura

---

El system prompt define el comportamiento del agente:

```
"Eres un asistente experto en motores de combustión que responde preguntas 
basándote en documentos específicos. Tu objetivo es proporcionar respuestas 
precisas y fundamentadas. Si la información no está disponible, lo indicarás 
claramente."
```

**Justificación de cada decisión:**

- **Rol específico** — El modelo sabe qué dominio cubre y no divaga en temas ajenos
- **"Basándote en documentos específicos"** — Ancla las respuestas al contexto recuperado, previene alucinaciones
- **"Si no está disponible, lo indica"** — Honestidad explícita sobre los límites del conocimiento

**Temperature = 0.0** (determinista)

Para un dominio técnico se prioriza la precisión sobre la creatividad. `temperature=0.0` hace que el modelo reproduzca la información de los documentos sin interpolar ni inventar.

---

## 🧠 Paso 7 — Memoria Conversacional

---

La memoria se implementa con `MemorySaver` de LangGraph y un identificador de sesión (`thread_id`).

```python
memoria = MemorySaver()
app = grafo.compile(checkpointer=memoria)

# Cada conversación tiene su propio ID
config = {"configurable": {"thread_id": "sesion-001"}}
resultado = app.invoke(estado, config=config)
```

**Cómo funciona:**

- Cada pregunta se añade al historial de mensajes del `thread_id`
- En la siguiente pregunta, el historial completo se incluye en el contexto del LLM
- Si cambia el `thread_id`, empieza una conversación nueva sin memoria de la anterior

**Ejemplo:**

```
Pregunta 1: "¿Cuáles son las fases del motor de 4 tiempos?"
Pregunta 2: "¿Y cuál es la diferencia con los de 2 tiempos?"
# El agente sabe que se refiere a "motores" sin que se lo digan
```

---

## 🔄 Paso 8 — Failover de Modelos Gemini

---

La API de Gemini en el plan gratuito tiene un límite de **20 peticiones por día por modelo**.

Para garantizar disponibilidad, el sistema implementa failover automático entre modelos:

```python
MODELOS_GEMINI = [
    "gemini-2.5-flash",   # Intenta primero
    "gemini-2.0-flash",   # Si falla, intenta este
    "gemini-1.5-pro"      # Si falla, intenta este
]
```

**Cuándo se activa:**

Si el modelo actual devuelve un error `429 RESOURCE_EXHAUSTED` (cuota excedida), el nodo de generación captura la excepción e intenta automáticamente con el siguiente modelo de la lista.

```
gemini-2.5-flash ❌ cuota agotada
    → gemini-2.0-flash ✅ responde
```

**El usuario no percibe el cambio.** Si todos los modelos agotan su cuota, el sistema devuelve un mensaje de error controlado en lugar de crashear.

---

## 🌐 Paso 9 — Interfaz Web con Streamlit

---

La interfaz se construyó con **Streamlit**, que permite crear aplicaciones web directamente desde Python.

**Decisiones técnicas clave:**

**`@st.cache_resource`** — El agente (ChromaDB + modelo HuggingFace + LangGraph) se carga una sola vez al arrancar la aplicación. Sin este decorador, Streamlit recargaría todos los modelos en cada interacción del usuario.

```python
@st.cache_resource
def cargar_agente():
    from agente_rag_langgraph_completo import app, vectorstore
    return app, vectorstore
```

**`st.session_state`** — Mantiene el historial del chat visible en pantalla entre interacciones, sin necesidad de base de datos externa.

**`thread_id` editable** — El usuario puede cambiar el identificador de sesión desde el sidebar para iniciar conversaciones independientes.

---

## 🎬 DEMO EN VIVO

---

> **📺 [CAMBIAR PANTALLA → Abrir Streamlit en localhost:8501]**

Vamos a ver el sistema funcionando. Haré 4 demostraciones:

**1. Respuesta precisa desde documentos**
> Escribir: *"¿Cómo funciona un motor de 4 tiempos?"*

El sistema busca en los 632 fragmentos, recupera los 3 más relevantes y genera la respuesta.

**2. Memoria conversacional**
> Escribir: *"¿Y cuál es la diferencia con los de 2 tiempos?"*

Sin mencionar "motor", el agente recuerda el contexto anterior.

**3. Honestidad del sistema**
> Escribir: *"¿Cuál es la capital de Francia?"*

El agente responde que no tiene esa información en sus documentos. No inventa.

**4. Fase técnica específica**
> Escribir: *"¿Qué es la fase de compresión?"*

Respuesta técnica precisa extraída de los manuales indexados.

> **📺 [VOLVER AL NOTEBOOK]**

---

## 🚀 Cierre

---

**Resumen técnico de lo que se ha construido:**

- **158 páginas** de documentación técnica procesadas y fragmentadas en 632 chunks
- **Embeddings locales** con HuggingFace — sin dependencia de APIs externas
- **ChromaDB** como base vectorial persistente con búsqueda semántica
- **Grafo LangGraph** con 2 nodos: retrieval + generación
- **System prompt justificado** y temperatura determinista (0.0)
- **Memoria conversacional** por thread_id con MemorySaver
- **Failover automático** entre 3 modelos Gemini
- **Interfaz Streamlit** con caché y gestión de sesión

**El conocimiento de una organización es su activo más valioso. Este sistema lo pone a disposición de todos, en segundos.**

Quedo abierto a preguntas.

---